In [1]:
import numpy as np
import toponetx.datasets.graph as graph
import torch

from topomodelx.nn.simplicial.hsn import HSN
from topomodelx.utils.sparse import from_sparse

%load_ext autoreload
%autoreload 2

In [2]:
dataset = graph.karate_club(complex_type="simplicial")
print(dataset)

Simplicial Complex with shape (34, 78, 45, 11, 2) and dimension 4


In [3]:
incidence_1 = dataset.incidence_matrix(rank=1)
adjacency_0 = dataset.adjacency_matrix(rank=0)

incidence_1 = from_sparse(incidence_1)
adjacency_0 = from_sparse(adjacency_0)

print(f"The incidence matrix B1 has shape: {incidence_1.shape}.")
print(f"The adjacency matrix A0 has shape: {adjacency_0.shape}.")

The incidence matrix B1 has shape: torch.Size([34, 78]).
The adjacency matrix A0 has shape: torch.Size([34, 34]).


In [5]:
x_0 = list(dataset.get_simplex_attributes("node_feat").values())
x_0 = torch.tensor(np.stack(x_0))
channels_nodes = x_0.shape[-1]

print(f"There are {x_0.shape[0]} nodes with features of dimension {x_0.shape[1]}.")

There are 34 nodes with features of dimension 2.


In [17]:
x_0.shape

torch.Size([34, 2])

In [6]:
y = np.array(
    [
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        0,
        1,
        1,
        1,
        1,
        0,
        0,
        1,
        1,
        0,
        1,
        0,
        1,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
    ]
)
y_true = np.zeros((34, 2))
y_true[:, 0] = y
y_true[:, 1] = 1 - y
y_train = y_true[:30]
y_test = y_true[-4:]

y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [12]:
y_test.shape

torch.Size([4, 2])

In [7]:
class Network(torch.nn.Module):
    def __init__(self, channels, out_channels, n_layers=2):
        super().__init__()
        self.base_model = HSN(
            channels=channels,
            n_layers=n_layers,
        )
        self.linear = torch.nn.Linear(channels, out_channels)

    def forward(self, x, incidence_1, adjacency_0):
        x = self.base_model(x, incidence_1, adjacency_0)
        x = self.linear(x)
        return torch.softmax(x, dim=1)

In [15]:
out_channels = 2
n_layers = 3

model = Network(
    channels=channels_nodes,
    out_channels=out_channels,
    n_layers=2,
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

In [16]:
test_interval = 10
num_epochs = 100
for epoch_i in range(1, num_epochs + 1):
    epoch_loss = []
    model.train()
    optimizer.zero_grad()

    y_hat = model(x_0, incidence_1, adjacency_0)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        y_hat[: len(y_train)].float(), y_train.float()
    )
    epoch_loss.append(loss.item())
    loss.backward()
    optimizer.step()

    y_pred = torch.where(y_hat > 0.5, torch.tensor(1), torch.tensor(0))
    accuracy = (y_pred[-len(y_train) :] == y_train).all(dim=1).float().mean().item()
    print(
        f"Epoch: {epoch_i} loss: {np.mean(epoch_loss):.4f} Train_acc: {accuracy:.4f}",
        flush=True,
    )
    if epoch_i % test_interval == 0:
        with torch.no_grad():
            y_hat_test = model(x_0, incidence_1, adjacency_0)
            y_pred_test = torch.where(
                y_hat_test > 0.5, torch.tensor(1), torch.tensor(0)
            )
            test_accuracy = (
                torch.eq(y_pred_test[-len(y_test) :], y_test)
                .all(dim=1)
                .float()
                .mean()
                .item()
            )
            print(f"Test_acc: {test_accuracy:.4f}", flush=True)

Epoch: 1 loss: 0.7214 Train_acc: 0.4000
Epoch: 2 loss: 0.7144 Train_acc: 0.5667
Epoch: 3 loss: 0.7068 Train_acc: 0.5667
Epoch: 4 loss: 0.6976 Train_acc: 0.5333
Epoch: 5 loss: 0.6881 Train_acc: 0.5333
Epoch: 6 loss: 0.6801 Train_acc: 0.5333
Epoch: 7 loss: 0.6742 Train_acc: 0.4667
Epoch: 8 loss: 0.6713 Train_acc: 0.5333
Epoch: 9 loss: 0.6669 Train_acc: 0.5333
Epoch: 10 loss: 0.6614 Train_acc: 0.5333
Test_acc: 0.0000
Epoch: 11 loss: 0.6568 Train_acc: 0.5333
Epoch: 12 loss: 0.6497 Train_acc: 0.5333
Epoch: 13 loss: 0.6453 Train_acc: 0.5667
Epoch: 14 loss: 0.6431 Train_acc: 0.5667
Epoch: 15 loss: 0.6388 Train_acc: 0.5667
Epoch: 16 loss: 0.6378 Train_acc: 0.5667
Epoch: 17 loss: 0.6350 Train_acc: 0.5667
Epoch: 18 loss: 0.6286 Train_acc: 0.5667
Epoch: 19 loss: 0.6298 Train_acc: 0.5667
Epoch: 20 loss: 0.6276 Train_acc: 0.5667
Test_acc: 0.0000
Epoch: 21 loss: 0.6238 Train_acc: 0.5667
Epoch: 22 loss: 0.6238 Train_acc: 0.5333
Epoch: 23 loss: 0.6184 Train_acc: 0.6000
Epoch: 24 loss: 0.6186 Train_acc